In [61]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.optim as optim

## ViT

In [62]:
# CIFAR-10 normalizasyon değerleri
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
data_path = r"C:\Users\oguzh\OneDrive\Masaüstü\data\CIFAR-10"
train_dataset = datasets.CIFAR10(
    root=data_path,
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [63]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self,embed_dim=64, num_heads=8, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim= embed_dim, num_heads= num_heads, batch_first= True)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden_dim = embed_dim * mlp_ratio

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, X):
        z = self.norm1(X)
        attention, _ = self.attention(z,z,z, need_weights= False)
        X = X + attention

        X = X + self.mlp(self.norm2(X))

        return X


In [64]:
class ViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=10,
        embed_dim=64,
        num_heads=8,
        depth=6
    ):
        super().__init__()
        assert img_size % patch_size == 0
        assert embed_dim % num_heads == 0

        num_patches = (img_size // patch_size) ** 2

        # 1. Patch embedding
        self.patch_embedding = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # 2. CLS ve positional embedding
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )
        self.pos_embedding = nn.Parameter(
            torch.zeros(1, num_patches + 1, embed_dim)
        )

        # Küçük, öğrenilebilir başlangıç değerleri
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        # 3. Transformer Encoder blokları
        self.blocks = nn.Sequential(*[
            TransformerEncoderBlock(embed_dim, num_heads)
            for _ in range(depth)
        ])

        # 4. Son normalizasyon ve sınıflandırıcı
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        # (B, 3, 224, 224) -> (B, 64, 14, 14)
        x = self.patch_embedding(x)

        # (B, 64, 14, 14) -> (B, 196, 64)
        x = x.flatten(2).transpose(1, 2)

        # CLS: (B, 1, 64)
        cls = self.cls_token.expand(B, -1, -1)

        # (B, 196, 64) -> (B, 197, 64)
        x = torch.cat([cls, x], dim=1)

        # Konum bilgisini ekle
        x = x + self.pos_embedding

        # Transformer Encoder
        x = self.blocks(x)

        # CLS token'ını seç: (B, 197, 64) -> (B, 64)
        x = self.norm(x[:, 0, :])

        # Sınıf skorları: (B, 64) -> (B, num_classes)
        x = self.head(x)

        return x

In [65]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ViT(
    img_size=32,
    patch_size=4,
    num_classes=10,
    embed_dim=128,
    num_heads=4,
    depth=6
).to(device)

print("Device:", device)
print("Parametre sayısı:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

Device: cuda
Parametre sayısı: 1205898


In [66]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

epochs = 20

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

In [67]:
for epoch in range(epochs):

    # -------- TRAIN --------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(
            device, non_blocking=True
        )
        labels = labels.to(
            device, non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        predictions = logits.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # -------- EVALUATION --------
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(
                device, non_blocking=True
            )
            labels = labels.to(
                device, non_blocking=True
            )

            with torch.autocast(
                device_type=device.type,
                enabled=(device.type == "cuda")
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            val_loss += loss.item() * images.size(0)

            predictions = logits.argmax(dim=1)
            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {val_loss:.4f} | "
        f"Test Acc: {val_acc:.2f}%"
    )

Epoch [1/20] | Train Loss: 1.7286 | Train Acc: 35.43% | Test Loss: 1.5029 | Test Acc: 45.25%
Epoch [2/20] | Train Loss: 1.4083 | Train Acc: 48.55% | Test Loss: 1.3022 | Test Acc: 52.66%
Epoch [3/20] | Train Loss: 1.2661 | Train Acc: 54.19% | Test Loss: 1.2396 | Test Acc: 55.50%
Epoch [4/20] | Train Loss: 1.1728 | Train Acc: 57.63% | Test Loss: 1.1266 | Test Acc: 58.93%
Epoch [5/20] | Train Loss: 1.1070 | Train Acc: 60.26% | Test Loss: 1.0700 | Test Acc: 61.20%
Epoch [6/20] | Train Loss: 1.0328 | Train Acc: 62.72% | Test Loss: 1.0393 | Test Acc: 62.23%
Epoch [7/20] | Train Loss: 0.9884 | Train Acc: 64.58% | Test Loss: 0.9449 | Test Acc: 65.83%
Epoch [8/20] | Train Loss: 0.9365 | Train Acc: 66.29% | Test Loss: 0.9467 | Test Acc: 66.01%
Epoch [9/20] | Train Loss: 0.8932 | Train Acc: 68.09% | Test Loss: 0.8731 | Test Acc: 68.76%
Epoch [10/20] | Train Loss: 0.8521 | Train Acc: 69.70% | Test Loss: 0.8773 | Test Acc: 68.50%
Epoch [11/20] | Train Loss: 0.8029 | Train Acc: 71.31% | Test Loss: 0